# Extracting Conflicts of Interest from Scientific Papers with Local LLMs

![](assets/paper.png)

## Get the Authors

In [1]:
authors = "Claus F Vogelmeier, Konstantinos Kostikas, Juanzhi Fang, Hengfeng Tian, Bethan Jones, Christopher Ll Morgan, Robert Fogel, Florian S Gutzwiller, Hui Cao"

![](assets/author.png)

## Get Conflict of Interest

In [2]:
coi_statement = "CFV reports research support from AstraZeneca, Boehringer Ingelheim, GlaxoSmithKline, Grifols, Novartis, Bayer Schering, MSD, Pfizer, consultancy from AstraZeneca, Boehringer Ingelheim, CSL Behring, GlaxoSmithKline, Grifols, Menarini, Novartis, Teva, Cipla, and honoraria from AstraZeneca, Boehringer Ingelheim, CSL Behring, GlaxoSmithKline, Grifols, Menarini, Mundipharma, Novartis, Teva, Cipla. KK was an employee of Novartis Pharma AG at the time of the conduct of this study. JF is an employee of Novartis Pharmaceuticals Corporation owning stocks through the employment. HT is employee of KMK Consulting Inc. and providing consulting to Novartis, the sponsor of the study. BJ is an employee of Pharmatelligence who received funding from Novartis to conduct analyses for this study. CM is a consultant of Pharmatelligence who received funding from Novartis to conduct analyses for this study. RF is an employee of Novartis Pharmaceuticals Corporation owning stocks through the employment. FSG is an employee of Novartis Pharma AG owning stocks through the employment. HC is an employee of Novartis Pharmaceuticals Corporation owning stocks through the employment."

![](assets/coi.png)

## Create a prompt

In [ ]:
prompt = f"""Extract conflicts of interest as a table with columns: author, company, conflict.
One row per author-company-conflict relationship. Use the author list to resolve initials.

authors: {authors}

conflict of interest statement: {coi_statement}"""


## Run through a local model

This cell sends the prompt to a local LLM running in LM Studio and stores the raw text response.

In [7]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
model = "meta-llama-3.1-8b-instruct"

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)

This cell parses the model's markdown table output into a pandas DataFrame and displays it.

In [8]:
import pandas as pd

raw = response.choices[0].message.content.strip()

# Parse markdown table rows, skip separator lines (---|---)
lines = [l for l in raw.splitlines() if l.startswith("|") and "---" not in l]
rows = [[cell.strip() for cell in l.strip("|").split("|")] for l in lines]

df = pd.DataFrame(rows[1:], columns=rows[0])
df


,Name of Author,Company,Reasons for Conflict
0,Claus F Vogelmeier,AstraZeneca,"Research support, consultancy, honoraria"
1,Claus F Vogelmeier,Boehringer Ingelheim,"Research support, consultancy, honoraria"
2,Claus F Vogelmeier,GlaxoSmithKline,"Research support, consultancy, honoraria"
3,Claus F Vogelmeier,Grifols,"Research support, consultancy, honoraria"
4,Claus F Vogelmeier,Novartis,
5,Claus F Vogelmeier,Bayer Schering,Research support
6,Claus F Vogelmeier,MSD,Research support
7,Claus F Vogelmeier,Pfizer,Research support
8,Konstantinos Kostikas,Novartis Pharma AG,Employee at the time of study
9,Juanzhi Fang,Novartis Pharmaceuticals Corporation,Employee owning stocks through employment
